In [1]:
library(data.table)
library(pROC)

Type 'citation("pROC")' for a citation.


Attaching package: ‘pROC’


The following objects are masked from ‘package:stats’:

    cov, smooth, var




In [2]:
ds <- fread("/home/luojiawei/inspire_benchmark_data/operation_.csv")

In [3]:
names(ds)

[1] "op_id"                                      
  [2] "subject_id"                                 
  [3] "hadm_id"                                    
  [4] "case_id"                                    
  [5] "opdate"                                     
  [6] "age"                                        
  [7] "sex"                                        
  [8] "weight"                                     
  [9] "height"                                     
 [10] "race"                                       
 [11] "asa"                                        
 [12] "emop"                                       
 [13] "department"                                 
 [14] "antype"                                     
 [15] "icd10_pcs"                                  
 [16] "orin_time"                                  
 [17] "orout_time"                                 
 [18] "opstart_time"                               
 [19] "opend_time"                                 
 [20] "admission_time"                             
 [21] "discharge_time"                             
 [22] "anstart_time"                               
 [23] "anend_time"                                 
 [24] "cpbon_time"                                 
 [25] "cpboff_time"                                
 [26] "icuin_time"                                 
 [27] "icuout_time"                                
 [28] "inhosp_death_time"                          
 [29] "bmi"                                        
 [30] "los"                                        
 [31] "op_duration"                                
 [32] "or_duration"                                
 [33] "an_duration"                                
 [34] "cpb_duration"                               
 [35] "icu_duration"                               
 [36] "have_icu"                                   
 [37] "have_cpb"                                   
 [38] "death_30d"                                  
 [39] "surgery_site"                               
 [40] "preop_essential_hypertension"               
 [41] "preop_coronary_heart_disease"               
 [42] "preop_congestive_heart_failure"             
 [43] "preop_atrial_fibrillation_and_flutter"      
 [44] "preop_abnormalities_of_heart_beat"          
 [45] "preop_diabetes_mellitus"                    
 [46] "preop_cerebral_infarction"                  
 [47] "preop_transient_cerebral_ischemic_attacks"  
 [48] "preop_emphysema_or_copd"                    
 [49] "preop_asthma"                               
 [50] "preop_acute_upper_respiratory_infections"   
 [51] "preop_acute_lower_respiratory_infections"   
 [52] "preop_abnormalities_of_breathing"           
 [53] "preop_malignant_neoplasms"                  
 [54] "preop_in_situ_neoplasms"                    
 [55] "preop_benign_neoplasms"                     
 [56] "preop_neoplasms_of_uncertain_behavior"      
 [57] "preop_chronic_kidney_disease"               
 [58] "preop_chronic_viral_hepatitis"              
 [59] "preop_liver_disease"                        
 [60] "preop_gastro_esophageal_reflux_disease"     
 [61] "preop_anemia"                               
 [62] "preop_disorder_of_thyroid"                  
 [63] "preop_crrt"                                 
 [64] "preop_ecmo"                                 
 [65] "preop_vent"                                 
 [66] "postop_cardiac_performance_index"           
 [67] "postop_cardiac_function_index"              
 [68] "postop_continuous_renal_replacement_therapy"
 [69] "postop_extracorporeal_membrane_oxygenation" 
 [70] "postop_ventilation"                         
 [71] "postop_acute_kidney_injury"                 
 [72] "have_aki"                                   
 [73] "postop_acute_liver_injury"                  
 [74] "have_ali"                                   
 [75] "postop_postoperative_complication"          
 [76] "postop_nervous_system_complication"         
 [77] "postop_digestive_system_complication"      

In [5]:
y_list <- c("death_30d","have_icu","have_aki","have_ali","postop_lung_complications","postop_stroke","postop_cardiac_complications")
x_list <- c('asa','sort_score','CCI_score','RCRI_score')

In [6]:
# 定义预测变量和目标变量列表
y_list <- c("death_30d","have_icu","have_aki","have_ali","postop_lung_complications","postop_stroke","postop_cardiac_complications")
x_list <- c('asa','sort_score','CCI_score','RCRI_score')

# 创建数据集分割
ds_tr <- ds[ds$dataset==1,]
ds_te <- ds[ds$dataset==2,] 

# 创建结果文件夹
dir.create("results", showWarnings = FALSE)

# 建立模型并生成预测结果
for (y_var in y_list) {
  for (x_var in x_list) {
    cat(paste0("处理: x = ", x_var, ", y = ", y_var, "\n"))
    
    # 1. 构建逻辑回归模型
    formula_str <- paste0(y_var, " ~ ", x_var)
    model <- glm(as.formula(formula_str), data = ds_tr, family = binomial)
    
    # 2. 在测试集上进行预测
    pred_prob <- predict(model, newdata = ds_te, type = "response")
    
    # 3. 准备结果数据框
    results <- data.frame(
      op_id = ds_te$op_id,
      y_true = ds_te[[y_var]],
      y_pred_prob_0 = pred_prob
    )

    # 4. 对结果数据框进行na.omit操作
    results <- na.omit(results)
    
    # 4. 保存结果到CSV文件
    output_file <- paste0("results/preop_", x_var, "_", y_var, ".csv")
    write.csv(results, file = output_file, row.names = FALSE)
    
    # 5. 计算ROC曲线下面积
    roc_obj <- roc(ds_te[[y_var]], pred_prob)
    auc_value <- auc(roc_obj)
    cat(paste0("  AUC: ", round(auc_value, 4), "\n"))
  }
}

cat("所有模型已完成，结果保存在results/目录下\n")

处理: x = asa, y = death_30d


Setting levels: control = 0, case = 1

Setting direction: controls < cases



  AUC: 0.8187
处理: x = sort_score, y = death_30d


Setting levels: control = 0, case = 1

Setting direction: controls < cases



  AUC: 0.894
处理: x = CCI_score, y = death_30d


...

  AUC: 0.7122
处理: x = CCI_score, y = postop_cardiac_complications


Setting levels: control = 0, case = 1

Setting direction: controls < cases



  AUC: 0.4802
处理: x = RCRI_score, y = postop_cardiac_complications


Setting levels: control = 0, case = 1

Setting direction: controls < cases



  AUC: 0.7866
所有模型已完成，结果保存在results/目录下
